In [1]:
import pandas as pd

In [2]:
# filepath_or_buffer
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/refs/heads/master/titanic.csv"

In [3]:
df = pd.read_csv(url) # buffer

In [4]:
df.tail(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [5]:
del df["PassengerId"]
del df["Ticket"]
del df["Cabin"]

In [6]:
df.tail(5)

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
886,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,13.00,S
887,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,30.00,S
888,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,23.45,S
889,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,30.00,C
890,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,7.75,Q


In [7]:
import re
title = []
for row in df["Name"]:
    pattern = r", ([A-Za-z ]+\.).*"
    title.append(re.search(pattern, row).group(1))

In [8]:
df["Title"] = title

In [9]:
del df["Name"]

In [10]:
df.tail(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
886,0,2,male,27.0,0,0,13.00,S,Rev.
887,1,1,female,19.0,0,0,30.00,S,Miss.
888,0,3,female,NaN,1,2,23.45,S,Miss.
889,1,1,male,26.0,0,0,30.00,C,Mr.
890,0,3,male,32.0,0,0,7.75,Q,Mr.


In [11]:
X = df.drop("Survived", axis = 1)
y = df["Survived"]

# Separar variables

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
# Separar conjuntos de datos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify = y)

In [14]:
X_train = X_train.reset_index(drop = True)
X_test = X_test.reset_index(drop = True)

In [15]:
X_train["Age"].mean()

np.float64(30.013846153846153)

In [16]:
# Imputación de variables
age_mean_pob = X_train["Age"].mean()
X_train["Age"] = X_train["Age"].fillna(age_mean_pob)
X_test["Age"] = X_test["Age"].fillna(age_mean_pob)

In [17]:
moda_embarked_pob = X_train["Embarked"].mode().iloc[0]
X_train["Embarked"] = X_train["Embarked"].fillna(moda_embarked_pob)
X_test["Embarked"] = X_test["Embarked"].fillna(moda_embarked_pob)

In [18]:
# Crear variables
X_train["Family_size"] = X_train["SibSp"] + X_train["Parch"] + 1
X_test["Family_size"] = X_test["SibSp"] + X_test["Parch"] + 1

X_train["Alone"] = X_train["Family_size"] == 1
X_test["Alone"] = X_test["Family_size"] == 1


X_train["Free_ticket"] = X_train["Fare"] == 0.0
X_test["Free_ticket"] = X_test["Fare"] == 0.0

In [19]:
from sklearn.preprocessing import OneHotEncoder

In [20]:
for column in ["Title", "Pclass", "Embarked", "Sex"]:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown = "ignore")
    encoder.fit(X_train[[column]])
    X_train = pd.concat([
        X_train.drop(column, axis = 1),
        pd.DataFrame(
            encoder.transform(X_train[[column]]),
            columns = map(lambda x: column + "_" + str(x), encoder.categories_[0])
        )],
        axis = 1
    )
    X_test = pd.concat([
        X_test.drop(column, axis = 1),
        pd.DataFrame(
            encoder.transform(X_test[[column]]),
            columns = map(lambda x: column + "_" + str(x), encoder.categories_[0])
        )],
        axis = 1
    )

In [26]:
from sklearn.preprocessing import RobustScaler
for numeric_column in ["Age", "SibSp", "Parch", "Fare", "Family_size"]:
    scaler = RobustScaler()
    scaler.fit(X_train[[numeric_column]])
    X_train[numeric_column] = scaler.transform(X_train[[numeric_column]])
    X_test[numeric_column] = scaler.transform(X_test[[numeric_column]])

In [28]:
X_train.shape

(623, 31)

In [29]:
X_test.shape

(268, 31)

In [ ]:
# 1. Diseñar la red neuronal
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(units = 1, activation = "sigmoid")
])

In [ ]:
# 2. Configurar la red neuronal
model.compile(
    loss = "binary_crossentropy",
    optimizer = tf.keras.optimizers.SGD(learning_rate = 0.1),
    metrics=[tf.keras.metrics.F1Score(threshold=0.5)]
)

In [22]:
# 3. Entrenar modelo
model.fit(X_train, y_train, epochs = 10, verbose = 1)

,Age,SibSp,Parch,Fare,Family_size,Alone,Free_ticket,Title_Capt.,Title_Col.,Title_Dr.,...,Title_Sir.,Title_the Countess.,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male
0,6.000000,0,1,33.000,2,False,False,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
1,20.000000,0,0,7.050,1,True,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
2,4.000000,1,1,16.700,3,False,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
3,30.013846,0,0,7.750,1,True,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,21.000000,2,2,34.375,5,False,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
618,36.000000,0,0,13.000,1,True,False,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
619,52.000000,0,0,13.500,1,True,False,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
620,30.013846,0,0,7.750,1,True,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
621,18.000000,0,0,7.750,1,True,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


In [23]:
enconder.transform(X_test[["Title"]])

NameError: name 'enconder' is not defined

In [ ]:
pd.get_dummies(X_train["Title"]).head(2)

In [ ]:
pd.get_dummies(X_test["Title"]).head(2)

In [ ]:
X_test.isna().sum()

In [ ]:
import re
cadena = 'Duff Gordon, Sir. Cosmo Edmund ("Mr Morgan")'
pattern = r", ([A-Za-z]+\.).*"
print(re.search(pattern, cadena).group(1))

In [ ]:
help(re.match)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Validar nulos en numéricos
df.count(axis = 0, numeric_only = True) < df.shape[0]

In [ ]:
# Porcentaje de filas nulas
df.isna().sum()[df.isna().sum() > 0] / df.shape[0] * 100

In [ ]:
# Opción 1: ELIMINAR - Una presencia alta de nulos usualmente > 15% se elimina *
del df["Cabin"]
# Opción 2: La falta de información es información (Crear valor sintético)
#df["Cabin"] = df["Cabin"].fillna("Unknow")

In [ ]:
df["Age"].hist()

In [ ]:
# Imputación
df["Age"] = df["Age"].fillna(df["Age"].median()).astype(int)

In [ ]:
df

In [ ]:
# Porcentaje de filas nulas
df.isna().sum()[df.isna().sum() > 0] / df.shape[0] * 100

In [ ]:
# Imputación con moda para categorias
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode().iloc[0])

In [ ]:
df["Name"].value_counts().plot.bar()

In [ ]:
df[df["Ticket"] == "347082"]

In [ ]:
# Eliminamos todo lo que se distribuye de forma uniforme
del df["PassengerId"]
del df["Name"]
del df["Ticket"]

In [ ]:
df

In [ ]:
df["Sex"].value_counts()

In [ ]:
df["Embarked"].value_counts()

In [ ]:
df[["Pclass"]].groupby("Pclass").count()

In [ ]:
clases = []
for i in df["Pclass"]:
    clases.append(i)
set(clases)

In [ ]:
categorical_columns = ["Pclass", "Sex", "Embarked"]
for column in categorical_columns:
    dummies = pd.get_dummies(df[column], prefix=column).astype(int)
    df = pd.concat([df, dummies], axis=1) # Por columnas (Biyección)
    del df[column]

In [ ]:
df["Family_size"] = df["SibSp"] + df["Parch"] + 1

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(
    df.corr(),
    annot=True,  # Muestra los valores dentro de cada celda
    fmt=".2f",  # Define el formato numérico (2 decimales)
    cmap="coolwarm",  # Paleta de colores
    vmin=-1,
    vmax=1,  # Fija la escala entre -1 y 1
)
plt.title("Matriz de Correlación")
plt.show()

In [ ]:
df.describe()

In [ ]:
# MinMax Scaler
temp = (df["Age"] - df["Age"].min(axis=0)) / (df["Age"].max(axis=0) - df["Age"].min(axis=0))
temp.describe()

In [ ]:
# MinMax Scaler
numerical_values = ["Age", "SibSp", "Parch", "Fare", "Family_size"]
for column in numerical_values:
    df[column] = (
        (df[column] - df[column].min(axis=0))
        / (df[column].max(axis=0)
        - df[column].min(axis=0))
    )

In [ ]:
df

In [ ]:
X = df.drop("Survived", axis = 1) # Devuelve un dataframe sin la columna
Y = df["Survived"]

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
pca.fit(X)

In [ ]:
X_pca = pca.transform(X)

In [ ]:
df_pca = pd.DataFrame(X_pca, columns=["PCA1", "PCA2"])
df_pca["Y"] = Y

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_pca,
    x="PCA1",
    y="PCA2",
    hue="Y",
    palette="viridis",
    alpha=0.8,
)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
plt.xlabel(f"PCA 1 ({var1:.1f}% varianza)")
plt.ylabel(f"PCA 2 ({var2:.1f}% varianza)")

plt.title("Visualización de PCA (2 Componentes)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()